# JetGuard - LSTM Hyperparameter Tuning

Tune one parameter at a time around the existing 50-cycle LSTM baseline. The existing engine-wise 80/20 train-validation split is reused. Scaling is fitted on training engines only; validation selects the configuration and the official test set is evaluated only after selection.

The selection metric is the NASA asymmetric RUL score (lower is better). RMSE, MAE, and MAPE are also reported. Test predictions are clipped to the training target range, 0-125 cycles.

In [1]:
from pathlib import Path
import copy
import json
import random
import time

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATASET_DIR = PROJECT_ROOT / "dataset"
REPORT_DIR = PROJECT_ROOT / "reports"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
WINDOW_LENGTH = 50
MAX_EPOCHS = 80
PATIENCE = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)

CONFIGURATIONS = [
    {"name": "baseline", "hidden_size": 64, "dropout": 0.0, "learning_rate": 1e-3, "batch_size": 256},
    {"name": "hidden_32", "hidden_size": 32, "dropout": 0.0, "learning_rate": 1e-3, "batch_size": 256},
    {"name": "dropout_0.2", "hidden_size": 64, "dropout": 0.2, "learning_rate": 1e-3, "batch_size": 256},
    {"name": "learning_rate_0.0005", "hidden_size": 64, "dropout": 0.0, "learning_rate": 5e-4, "batch_size": 256},
    {"name": "batch_size_128", "hidden_size": 64, "dropout": 0.0, "learning_rate": 1e-3, "batch_size": 128},
]
print("Device:", DEVICE)
print("Configurations:", len(CONFIGURATIONS))

Device: cpu
Configurations: 5


In [2]:
with (PROCESSED_DIR / "feature_metadata.json").open("r", encoding="utf-8") as file:
    feature_metadata = json.load(file)
train_df = pd.read_csv(PROCESSED_DIR / "train_FD001_processed.csv")
test_columns = ["engine_id", "cycle"] + [f"setting_{i}" for i in range(1, 4)] + [f"sensor_{i}" for i in range(1, 22)]
test_df = pd.read_csv(DATASET_DIR / "test_FD001.txt", sep=r"\s+", header=None, names=test_columns)
test_rul = pd.read_csv(DATASET_DIR / "RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL_true"])["RUL_true"].to_numpy()

training_engines = set(feature_metadata["training_engines"])
validation_engines = set(feature_metadata["validation_engines"])
sensors = feature_metadata["informative_sensors"]
feature_columns = ["setting_1", "setting_2", "setting_3"] + sensors
train_split = train_df[train_df.engine_id.isin(training_engines)].copy()
validation_split = train_df[train_df.engine_id.isin(validation_engines)].copy()
assert training_engines.isdisjoint(validation_engines)
assert len(test_rul) == test_df.engine_id.nunique()

scaler = StandardScaler().fit(train_split[feature_columns])
def scale_frame(frame):
    result = frame.sort_values(["engine_id", "cycle"]).copy()
    scaled = scaler.transform(result[feature_columns])
    for index, column in enumerate(feature_columns):
        result[column] = scaled[:, index]
    return result

train_scaled = scale_frame(train_split)
validation_scaled = scale_frame(validation_split)
test_scaled = scale_frame(test_df)
joblib.dump(scaler, ARTIFACT_DIR / "lstm_tuned_standard_scaler.joblib")
print("Engines:", train_split.engine_id.nunique(), "train /", validation_split.engine_id.nunique(), "validation /", test_df.engine_id.nunique(), "test")

Engines: 80 train / 20 validation / 100 test


In [3]:
def make_training_windows(frame):
    sequences, targets = [], []
    for _, engine in frame.groupby("engine_id", sort=True):
        engine = engine.sort_values("cycle")
        values = engine[feature_columns].to_numpy(dtype=np.float32)
        rul = engine.RUL_capped.to_numpy(dtype=np.float32)
        for end in range(WINDOW_LENGTH - 1, len(engine)):
            start = end - WINDOW_LENGTH + 1
            sequences.append(values[start:end + 1])
            targets.append(rul[end])
    return np.stack(sequences), np.asarray(targets, dtype=np.float32)


def make_test_windows(frame):
    sequences, engine_ids, final_cycles = [], [], []
    for engine_id, engine in frame.groupby("engine_id", sort=True):
        engine = engine.sort_values("cycle")
        values = engine[feature_columns].to_numpy(dtype=np.float32)
        if len(values) < WINDOW_LENGTH:
            padding = np.repeat(values[:1], WINDOW_LENGTH - len(values), axis=0)
            values = np.concatenate([padding, values], axis=0)
        sequences.append(values[-WINDOW_LENGTH:])
        engine_ids.append(int(engine_id))
        final_cycles.append(int(engine.cycle.iloc[-1]))
    return np.stack(sequences), np.asarray(engine_ids), np.asarray(final_cycles)


x_train, y_train = make_training_windows(train_scaled)
x_validation, y_validation = make_training_windows(validation_scaled)
x_test, test_engine_ids, test_final_cycles = make_test_windows(test_scaled)
print("Sequence arrays:", x_train.shape, x_validation.shape, x_test.shape)

Sequence arrays: (12859, 50, 18) (2872, 50, 18) (100, 50, 18)


In [4]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class TunableLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, dropout):
        super().__init__()
        self.recurrent = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.regression = nn.Linear(hidden_size, 1)

    def forward(self, sequence):
        outputs, _ = self.recurrent(sequence)
        return self.regression(self.dropout(outputs[:, -1])).squeeze(1)


def nasa_score(y_true, y_pred):
    error = np.asarray(y_pred) - np.asarray(y_true)
    penalty = np.where(error < 0, np.exp(-error / 13.0) - 1.0, np.exp(error / 10.0) - 1.0)
    return float(penalty.sum())


def calculate_metrics(y_true, raw_prediction):
    y_true = np.asarray(y_true, dtype=float)
    prediction = np.clip(np.asarray(raw_prediction, dtype=float), 0.0, 125.0)
    nonzero = np.abs(y_true) > 1e-8
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, prediction))),
        "MAE": float(mean_absolute_error(y_true, prediction)),
        "MAPE": float(mean_absolute_percentage_error(y_true[nonzero], prediction[nonzero])),
        "NASA_score": nasa_score(y_true, prediction),
    }, prediction

In [5]:
def train_configuration(config):
    seed_everything(SEED)
    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
        batch_size=config["batch_size"], shuffle=True,
    )
    validation_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_validation), torch.from_numpy(y_validation)),
        batch_size=config["batch_size"], shuffle=False,
    )
    model = TunableLSTM(len(feature_columns), config["hidden_size"], config["dropout"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    loss_function = nn.MSELoss()
    best_state, best_loss, best_epoch, patience_count = None, float("inf"), 0, 0

    started = time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            loss_function(model(batch_x), batch_y).backward()
            optimizer.step()
        model.eval()
        validation_losses = []
        with torch.no_grad():
            for batch_x, batch_y in validation_loader:
                batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
                validation_losses.append(loss_function(model(batch_x), batch_y).item())
        validation_loss = float(np.mean(validation_losses))
        print(f"{config['name']} epoch={epoch} validation_mse={validation_loss:.3f}")
        if validation_loss < best_loss:
            best_loss, best_epoch = validation_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                break

    if best_state is None:
        raise RuntimeError(f"No checkpoint created for configuration {config['name']}.")
    model.load_state_dict(best_state)
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch_x, _ in validation_loader:
            predictions.extend(model(batch_x.to(DEVICE)).cpu().numpy())
    elapsed = time.perf_counter() - started
    return model, np.asarray(predictions), best_epoch, best_loss, elapsed

In [6]:
tuning_results = []
trained_models = {}
for config in CONFIGURATIONS:
    print("\nTraining", config["name"])
    model, raw_predictions, best_epoch, best_mse, elapsed = train_configuration(config)
    validation_metrics, validation_predictions = calculate_metrics(y_validation, raw_predictions)
    tuning_results.append({
        **config,
        "best_epoch": best_epoch,
        "best_validation_mse": best_mse,
        "training_seconds": elapsed,
        **validation_metrics,
    })
    trained_models[config["name"]] = model

tuning_results_df = pd.DataFrame(tuning_results).sort_values("NASA_score").reset_index(drop=True)
tuning_results_df.to_csv(REPORT_DIR / "lstm_hyperparameter_validation.csv", index=False)
tuning_results_df


Training baseline


baseline epoch=1 validation_mse=5471.258


baseline epoch=2 validation_mse=4893.715


baseline epoch=3 validation_mse=4479.684


baseline epoch=4 validation_mse=4125.387


baseline epoch=5 validation_mse=3810.172


... (326 outputs left out)

batch_size_128 epoch=41 validation_mse=176.173


batch_size_128 epoch=42 validation_mse=161.186


batch_size_128 epoch=43 validation_mse=184.464


,name,hidden_size,dropout,learning_rate,batch_size,best_epoch,best_validation_mse,training_seconds,RMSE,MAE,MAPE,NASA_score
0,baseline,64,0.0,0.0010,256,55,151.115612,75.364608,12.673987,9.813985,0.167465,6240.983626
1,dropout_0.2,64,0.2,0.0010,256,60,150.561509,82.890241,12.659446,9.529096,0.158396,7456.503149
2,hidden_32,32,0.0,0.0010,256,80,224.635360,50.781244,15.442757,11.718747,0.175682,8166.951323
3,batch_size_128,64,0.0,0.0010,128,35,156.182777,62.057100,12.613948,9.481206,0.175841,8399.724054
4,learning_rate_0.0005,64,0.0,0.0005,256,80,243.589411,105.744704,16.071451,12.045823,0.175266,8693.961867


In [7]:
selected_result = tuning_results_df.iloc[0]
selected_name = str(selected_result["name"])
selected_config = next(config for config in CONFIGURATIONS if config["name"] == selected_name)
selected_model = trained_models[selected_name]

selected_model.eval()
with torch.no_grad():
    raw_test_predictions = selected_model(torch.from_numpy(x_test).to(DEVICE)).cpu().numpy()
test_metrics, test_predictions = calculate_metrics(test_rul, raw_test_predictions)

test_output = pd.DataFrame({
    "engine_id": test_engine_ids,
    "final_cycle": test_final_cycles,
    "true_RUL": test_rul,
    "predicted_RUL": test_predictions,
})
test_output.to_csv(REPORT_DIR / "lstm_tuned_test_predictions.csv", index=False)
pd.DataFrame([{"model": "Tuned LSTM", "configuration": selected_name, **test_metrics}]).to_csv(
    REPORT_DIR / "lstm_tuned_test_metrics.csv", index=False
)
torch.save(selected_model.state_dict(), ARTIFACT_DIR / "lstm_tuned_selected.pth")

metadata = {
    "selection_metric": "validation NASA_score (lower is better)",
    "window_length": WINDOW_LENGTH,
    "selected_configuration": selected_config,
    "best_epoch": int(selected_result["best_epoch"]),
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "seed": SEED,
    "device": str(DEVICE),
    "train_engines": sorted(training_engines),
    "validation_engines": sorted(validation_engines),
    "official_test_metrics": test_metrics,
}
(ARTIFACT_DIR / "lstm_tuning_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Selected validation configuration:", selected_config)
print("Official test metrics:", json.dumps(test_metrics, indent=2))
test_output.head()

Selected validation configuration: {'name': 'baseline', 'hidden_size': 64, 'dropout': 0.0, 'learning_rate': 0.001, 'batch_size': 256}
Official test metrics: {
  "RMSE": 16.860206101677257,
  "MAE": 12.675821590423585,
  "MAPE": 0.1807665238874871,
  "NASA_score": 407.59575374597205
}


,engine_id,final_cycle,true_RUL,predicted_RUL
0,1,31,112,107.315567
1,2,49,98,106.207825
2,3,126,69,59.541710
3,4,106,82,74.623276
4,5,98,91,74.844330


## Tuning scope

This is a small, one-factor-at-a-time search around the existing LSTM setup, not an exhaustive optimizer. The 50-cycle window was fixed based on the previous validation experiment. Hidden size, dropout, learning rate, and batch size were tested against the same validation split. The test set is used only after choosing the lowest validation NASA-score configuration. For stronger claims, repeat the selected configuration within engine-grouped cross-validation.